# TalkNet-ASD: End-to-End Inference Demo
This notebook runs the TalkNet-ASD Active Speaker Detection pipeline on Google Colab (GPU).

**Instructions:**
1. Make sure you are using a GPU runtime: `Runtime -> Change runtime type -> T4 GPU`.
2. Run the cells below sequentially.

We use TWO test videos:
- **001.mp4**: The repo's own built-in demo (guaranteed to work perfectly)
- **podcast_clip.mp4**: A real podcast/interview clip downloaded from YouTube to simulate Marvedge's actual use case (webinars, podcasts with speaker turns)

In [ ]:
# 1. Setup Environment
!git clone https://github.com/TaoRuijie/TalkNet-ASD.git
%cd TalkNet-ASD

# Install dependencies
!pip install scenedetect==0.5.6.1
!pip install gdown scipy librosa opencv-python python_speech_features tqdm

# Install ffmpeg and yt-dlp
!apt-get update -qq && apt-get install -y -qq ffmpeg
!pip install yt-dlp

# CRITICAL PATCH: Fix deprecated np.int/np.float/np.bool for NumPy 1.24+
!find . -name "*.py" -exec sed -i "s/np\.int)/int)/g" {} +
!find . -name "*.py" -exec sed -i "s/np\.int,/int,/g" {} +
!find . -name "*.py" -exec sed -i "s/np\.int]/int]/g" {} +
!find . -name "*.py" -exec sed -i "s/np\.float)/float)/g" {} +
!find . -name "*.py" -exec sed -i "s/np\.bool)/bool)/g" {} +

print("\n=== Environment setup complete ===")

In [ ]:
# 2. Download Test Videos

# --- Video A: The repo's own demo video (001.mp4) ---
# This is already bundled in the demo/ folder by the repo itself.
# It's the gold-standard test that the authors designed for.
!ls -lh demo/001.mp4 || echo "WARNING: demo/001.mp4 not found, the repo may have changed."

# --- Video B: A real podcast/interview clip from YouTube ---
# We download a 2-minute segment of a multi-person podcast interview.
# This simulates Marvedge's real use case: webinars and talking-head content.
!yt-dlp -f "bv*[ext=mp4]+ba[ext=m4a]/b[ext=mp4]" \
    --download-sections "*00:00:30-00:02:00" \
    --force-keyframes-at-cuts \
    -o "demo/podcast_clip.%(ext)s" \
    "https://www.youtube.com/watch?v=DxREm3s1scA" 2>/dev/null || \\
    echo "YouTube download failed. Using only the built-in demo video."

!ls -lh demo/*.mp4
print("\n=== Videos ready ===")

In [ ]:
# 3A. Run Inference on the BUILT-IN demo video (001.mp4)
# This is guaranteed to produce perfect results with speaker tracking.
# The pretrained checkpoint downloads automatically on first run.

!python demoTalkNet.py --videoName 001
print("\n=== Inference on 001.mp4 complete ===")

In [ ]:
# 3B. Run Inference on the podcast clip (if downloaded)
import os
if os.path.exists("demo/podcast_clip.mp4"):
    !python demoTalkNet.py --videoName podcast_clip
    print("\n=== Inference on podcast_clip complete ===")
else:
    print("Podcast clip was not downloaded. Skipping.")

In [ ]:
# 4. View and Download Results
import os

results = {
    "Built-in Demo (001)": "demo/001/pyavi/video_out.avi",
    "Podcast Clip": "demo/podcast_clip/pyavi/video_out.avi"
}

for name, path in results.items():
    if os.path.exists(path):
        size = os.path.getsize(path) / (1024*1024)
        print(f"\n=== {name}: SUCCESS ({size:.1f} MB) ===")
        print(f"  Output: {path}")
    else:
        print(f"\n=== {name}: NOT FOUND ===")
        # Check what IS there
        parent = os.path.dirname(path)
        if os.path.exists(parent):
            print(f"  Contents of {parent}:")
            for f in os.listdir(parent):
                print(f"    {f}")

print("\n--- Download the .avi files from the Colab sidebar (left panel > Files) ---")

In [ ]:
# 5. Auto-download result files to your local machine
from google.colab import files
import os

for path in ["demo/001/pyavi/video_out.avi", "demo/podcast_clip/pyavi/video_out.avi"]:
    if os.path.exists(path):
        print(f"Downloading {path}...")
        files.download(path)